# Exploración de variables: Tarea 3
**Introducción a Ciencia de Datos para Economía** - Equipo 2

El objetivo de este cuaderno de trabajo es poder explorar cómo se encuentra la base de datos de incidentes viales del C5 (CDMX), ofrecer una limpieza de esta base de datos y algo del análisis exploratorio de la misma.

## Carga de la base de datos

Cargamos la base de datos que proviente de [Incidentes viales reportados por C5](https://datos.cdmx.gob.mx/dataset/incidentes-viales-c5).

In [21]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/ElenaVillano/databases/refs/heads/main/db_falsas_alarmas.csv"
df = pd.read_csv(url)

## 1.- Estructura de la base de datos

En esta sección se realiza una exploración inicial de la base de datos de incidentes reportados al C5 de la Ciudad de México

In [22]:
print(df.head())

               folio  fecha_hora_creacion    fecha_hora_cierre  \
0  C5/20220923/04016  2022-09-23 18:38:34  2022-09-23 22:22:27   
1  C5/20220829/00630  2022-08-29 02:56:31  2022-08-29 06:04:53   
2  C5/20220724/01907  2022-07-24 04:29:09  2022-07-24 07:36:49   
3  C5/20240215/03071  2024-02-15 16:20:03  2024-02-15 20:52:19   
4  C5/20221006/03493  2022-10-06 19:33:05  2022-10-06 22:37:08   

  tipo_incidente_c4           incidente_c4         alcaldia_inicio  \
0         Accidente  Choque sin lesionados     VENUSTIANO CARRANZA   
1         Accidente           Motociclista              CUAUHTEMOC   
2         Accidente  Choque sin lesionados              IZTAPALAPA   
3         Accidente  Choque sin lesionados  LA MAGDALENA CONTRERAS   
4         Accidente  Choque sin lesionados              IZTAPALAPA   

  codigo_cierre  clas_con_f_alarma     tipo_entrada         alcaldia_cierre  \
0             A         EMERGENCIA  LLAMADA DEL 911     VENUSTIANO CARRANZA   
1             A  URGENCI

In [23]:
# Dimensiones de la base de datos
print("Número de registros:", df.shape[0])
print("Número de variables:", df.shape[1])

# Nombres de las variables
print("\nNombres de las variables:")
for i, columna in enumerate(df.columns, start=1):
    print(f"{i}. {columna}")

Número de registros: 124261
Número de variables: 14

Nombres de las variables:
1. folio
2. fecha_hora_creacion
3. fecha_hora_cierre
4. tipo_incidente_c4
5. incidente_c4
6. alcaldia_inicio
7. codigo_cierre
8. clas_con_f_alarma
9. tipo_entrada
10. alcaldia_cierre
11. alcaldia_catalogo
12. colonia_catalogo
13. longitud
14. latitud


Con la información anterior, vemos que la base de datos se compone de 124,261 registros de 12 variables, las cuales son: 
1. folio
2. fecha_hora_creacion
3. fecha_hora_cierre
4. tipo_incidente_c4
5. incidente_c4
6. alcaldia_inicio
7. codigo_cierre
8. clas_con_f_alarma
9. tipo_entrada
10. alcaldia_cierre
11. alcaldia_catalogo
12. colonia_catalogo
13. longitud
14. latitud

Ahora, revisamos los tipos de datos que tenemos disponibles de la base de datos cruda:

In [24]:
df.dtypes

folio                      str
fecha_hora_creacion        str
fecha_hora_cierre          str
tipo_incidente_c4          str
incidente_c4               str
alcaldia_inicio            str
codigo_cierre              str
clas_con_f_alarma          str
tipo_entrada               str
alcaldia_cierre            str
alcaldia_catalogo          str
colonia_catalogo           str
longitud               float64
latitud                float64
dtype: object

Vemos que, de las 14 variables, 12 vienen como texto y 2 de ellas vienen como número flotante. Por lo que se observa, habrá que hacer una limpieza para los elementos temporales que se incluyen.

Separamos las variables numéricas de las categóricas para poder realizar los análisis subsecuentes:

In [25]:
# Variables numéricas: pandas las detecta como int o float
vars_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Variables categóricas: pandas las detecta como 'str' 
vars_categoricas = df.select_dtypes(include='str').columns.tolist()

In [26]:
vars_categoricas

['folio',
 'fecha_hora_creacion',
 'fecha_hora_cierre',
 'tipo_incidente_c4',
 'incidente_c4',
 'alcaldia_inicio',
 'codigo_cierre',
 'clas_con_f_alarma',
 'tipo_entrada',
 'alcaldia_cierre',
 'alcaldia_catalogo',
 'colonia_catalogo']

In [27]:
vars_numericas

['longitud', 'latitud']

In [28]:
def analizar_categoricas(df, columnas):
    
    resultados = []

    for columna in columnas:

        frecuencias = df[columna].value_counts()

        resultado = {
            "variable": columna,
            "n_categorias": df[columna].nunique(),
            "n_faltantes": df[columna].isnull().sum(),
            "moda": frecuencias.index[0],
            "top_2": frecuencias.index[1] if len(frecuencias) > 1 else "-",
            "top_3": frecuencias.index[2] if len(frecuencias) > 2 else "-"
        }

        resultados.append(resultado)

    return pd.DataFrame(resultados)

In [29]:
analizar_categoricas(df, vars_categoricas)

,variable,n_categorias,n_faltantes,moda,top_2,top_3
0,folio,124261,0,C5/20220923/04016,C5/20220829/00630,C5/20220724/01907
1,fecha_hora_creacion,124106,0,2023-10-21 14:45:51,2023-12-03 17:42:18,2022-04-10 17:30:28
2,fecha_hora_cierre,107462,0,2023-11-04 19:33:22,2023-10-30 15:02:48,2023-12-26 20:02:40
3,tipo_incidente_c4,7,0,Accidente,Lesionado,Cadáver
4,incidente_c4,16,0,Choque sin lesionados,Choque con lesionados,Atropellado
5,alcaldia_inicio,16,8,IZTAPALAPA,GUSTAVO A. MADERO,CUAUHTEMOC
6,codigo_cierre,4,0,D,A,F
7,clas_con_f_alarma,5,0,URGENCIAS MEDICAS,EMERGENCIA,DELITO
8,tipo_entrada,7,0,LLAMADA DEL 911,BOTÓN DE AUXILIO,RADIO
9,alcaldia_cierre,16,8,IZTAPALAPA,GUSTAVO A. MADERO,CUAUHTEMOC


De esta tabla podemos observar que 8 de los registros no contienen alcaldía de inicio ni alcaldía de cierre; además, 2,718 registros no contienen una colonia seleccionada del catálogo.

In [30]:
df[vars_numericas].describe()

,longitud,latitud
count,124261.000000,124261.000000
mean,-99.134031,19.381444
std,0.061571,0.071538
min,-99.352914,19.094877
25%,-99.174500,19.332895
50%,-99.136941,19.383000
75%,-99.092380,19.434346
max,-98.947940,19.632201


Para las variables numéricas, se aprecia que estas corresponden únicamente a valores de geolocalización, por lo que la interpretación de sus estadísticos (como la media) no son relevantes para el análisis.

In [31]:
df.groupby("codigo_cierre")

## 2.- Limpieza

Para la limpieza de la base de datos, hacemos lo siguiente:

1. Primero un cambio en los nombres de las columnas para mantener una estructura de letras minúsculas y sin espacios.
2. Establecemos las variables de fecha en formato YYYY/MM/DD HH:mm:ss
3. Creamos la variable `duración_h` para tener el tiempo transcurrido entre que se creó y se cerró el informe
4. Las variables de alcaldía las pasamos a mayúsculas sin acentos.

In [38]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

FORMATO = "%Y-%m-%d %H:%M:%S"

df["fecha_hora_creacion"] = pd.to_datetime(df["fecha_hora_creacion"], format=FORMATO)
df["fecha_hora_cierre"] = pd.to_datetime(df["fecha_hora_cierre"], format=FORMATO)
df["duracion_h"] = (df["fecha_hora_cierre"] - df["fecha_hora_creacion"]).dt.total_seconds() / 3600

columnas_alcaldia = ["alcaldia_inicio", "alcaldia_cierre", "alcaldia_catalogo"]

for col in columnas_alcaldia:
    df[col] = (
        df[col].str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("ascii").str.upper().str.strip()
        )

## ¿Qué fenómeno registra la base? 

Las llamadas recibidas al Centro de Comando, Control, Cómputo, Comunicaciones y Contacto Ciudadano de la Ciudad de México (C5) por incidentes viales regresitrados e la Ciudad de México.  

**Entidad**: un folio, es decir, un reporte de incidente vial que el C5 abrió y cerró.
**Eventos**: la apertura del folio, cuando el C5 recibe el aviso y el cierre del folio, con el `codigo_cierre` que dice qué resultó.

## ¿Quién la produjo y con qué propósito? 

El gobierno de la Ciudad de México, con el objetivo de identificar los accidentes viales más frecuentes y los lugares donde más se producen.

## ¿Cuántas filas y columnas tiene?

In [33]:
# Dimensiones de la base de datos
df.shape

(124261, 15)

La base original tiene 124,261 y 14 variables. 

## ¿Qué significa cada variable? 

- folio: Número de identificación de la llamada al C5.
- fecha_hora_creacion: Hora de recepción de la llamada.
- fecha_hora_cierre: Hora en que se termino de atender la emergencia reportada.
- tipo de incidente_c4: Naturaleza de la emergencia reportada.
- incidente_c4: Espicifación de los involucrados en el tipo de incidente.
- alcaldia_inicio: Alcaldía donde se reportó e inició el seguimiento del incidente.
- codigo_cierre: Identificador asignado cuando se termina la atención a la emergencia, dependiendo si fue un caso falso o no.
- clas_con_f_alarma: Clasificación del tipo de emergencia.
- tipo_entrada: Medio por el cual se recibió la notificación de la emergencia.
- alcaldia_cierre: Alcaldía donde se terminó el seguimiento de la emergencia reportada.
- alcaldia_catalogo: Alcaldía donde sucedió la emergencia.
- colonia_catalogo: Colonia donde suceció la emergencia.
- longitud: Punto georeferencial del incidente.
- latitud: Punto georeferencial del incidente.

## Identifiquen tipos de datos y valores faltantes.

In [34]:
df.dtypes

folio                             str
fecha_hora_creacion    datetime64[us]
fecha_hora_cierre      datetime64[us]
tipo_incidente_c4                 str
incidente_c4                      str
alcaldia_inicio                   str
codigo_cierre                     str
clas_con_f_alarma                 str
tipo_entrada                      str
alcaldia_cierre                   str
alcaldia_catalogo                 str
colonia_catalogo                  str
longitud                      float64
latitud                       float64
duracion_h                    float64
dtype: object

In [35]:
df.isna().sum()

folio                     0
fecha_hora_creacion       0
fecha_hora_cierre         0
tipo_incidente_c4         0
incidente_c4              0
alcaldia_inicio           8
codigo_cierre             0
clas_con_f_alarma         0
tipo_entrada              0
alcaldia_cierre           8
alcaldia_catalogo       130
colonia_catalogo       2718
longitud                  0
latitud                   0
duracion_h                0
dtype: int64

Identifiquen posibles duplicados, categorías inconsistentes o coordenadas fuera del rango esperado.

In [ ]:
duplicados = df.duplicated()

print("Total de registros:", len(df))
print("Registros completamente duplicados:", duplicados.sum())
print("Porcentaje de duplicados:", round(duplicados.mean() * 100, 2), "%")

# Mostrar duplicados
display(df[duplicados].head(10))

Total de registros: 124261
Registros completamente duplicados: 0
Porcentaje de duplicados: 0.0 %


,folio,fecha_hora_creacion,fecha_hora_cierre,tipo_incidente_c4,incidente_c4,alcaldia_inicio,codigo_cierre,clas_con_f_alarma,tipo_entrada,alcaldia_cierre,alcaldia_catalogo,colonia_catalogo,longitud,latitud,fecha_hora_creacion


En esta base, no vemos registros duplicados.

In [ ]:
lat = pd.to_numeric(df["latitud"], errors="coerce")
lon = pd.to_numeric(df["longitud"], errors="coerce")

fuera_rango = (lat.notna() & lon.notna() & (~lat.between(19.0, 19.6) | ~lon.between(-99.4, -98.9)))

print("Coordenadas fuera del rango:", fuera_rango.sum())

display(df.loc[fuera_rango].head(10))

Coordenadas fuera del rango: 4


,folio,fecha_hora_creacion,fecha_hora_cierre,tipo_incidente_c4,incidente_c4,alcaldia_inicio,codigo_cierre,clas_con_f_alarma,tipo_entrada,alcaldia_cierre,alcaldia_catalogo,colonia_catalogo,longitud,latitud,duracion_h
26129,C5/220401/01123,2022-04-01 17:56:02,2022-04-01 17:59:04,Mi Calle,Incidente de tránsito,NaN,F,FALSA ALARMA,APLICATIVOS,NaN,NaN,NaN,-99.021432,19.632201,0.050556
40950,C5/220401/01134,2022-04-01 17:56:25,2022-04-01 18:00:22,Mi Calle,Incidente de tránsito,NaN,F,FALSA ALARMA,APLICATIVOS,NaN,NaN,NaN,-99.021432,19.632201,0.065833
59997,C5/220401/01137,2022-04-01 17:56:28,2022-04-01 17:59:48,Mi Calle,Incidente de tránsito,NaN,F,FALSA ALARMA,APLICATIVOS,NaN,NaN,NaN,-99.021432,19.632201,0.055556
108069,C5/220401/01124,2022-04-01 17:56:05,2022-04-01 17:58:30,Mi Calle,Incidente de tránsito,NaN,F,FALSA ALARMA,APLICATIVOS,NaN,NaN,NaN,-99.021432,19.632201,0.040278


Observamos que existen 4 registros con coordenadas fuera del rango esperado. Estos 4 registros además coinciden con registros que no tienen alcaldía registrada y que se catalogaron como Falsa Alarma.

Revisen si las variables están representadas correctamente, especialmente fechas y horas.


In [ ]:
df[['fecha_hora_creacion', 'fecha_hora_cierre']].sample(10, random_state=123)

En un paso anterior, realizamos el cambio de tipo de dato de cadena de texto a fecha y hora.

## Data Profiling o perfilado por tipo de variable

In [39]:
vars_categoricas = ["codigo_cierre", "tipo_incidente_c4", "incidente_c4", "clas_con_f_alarma",
                    "tipo_entrada", "alcaldia_inicio", "alcaldia_catalogo", "colonia_catalogo"]


def analizar_categoricas(df, columnas):
    resultados = []
    for columna in columnas:
        frecuencias = df[columna].value_counts()
        pct = frecuencias / frecuencias.sum() * 100

        resultados.append({
            "variable": columna,
            "n_categorias": df[columna].nunique(),
            "n_faltantes": df[columna].isnull().sum(),
            "% faltantes": round(df[columna].isnull().mean() * 100, 2),
            "moda": frecuencias.index[0],
            "% moda": round(pct.iloc[0], 1),
            "top_2": frecuencias.index[1],
            "top_3": frecuencias.index[2],
            "% top 3": round(pct.iloc[:3].sum(), 1),
            "categorías <1%": int((pct < 1).sum()),
        })
    return pd.DataFrame(resultados)


analizar_categoricas(df, vars_categoricas)

,variable,n_categorias,n_faltantes,% faltantes,moda,% moda,top_2,top_3,% top 3,categorías <1%
0,codigo_cierre,4,0,0.00,D,42.6,A,F,99.3,1
1,tipo_incidente_c4,7,0,0.00,Accidente,89.2,Lesionado,Cadáver,99.8,5
2,incidente_c4,16,0,0.00,Choque sin lesionados,45.7,Choque con lesionados,Atropellado,85.4,10
3,clas_con_f_alarma,5,0,0.00,URGENCIAS MEDICAS,51.0,EMERGENCIA,DELITO,100.0,3
4,tipo_entrada,7,0,0.00,LLAMADA DEL 911,89.6,BOTÓN DE AUXILIO,RADIO,99.5,4
5,alcaldia_inicio,16,8,0.01,IZTAPALAPA,16.3,GUSTAVO A. MADERO,CUAUHTEMOC,37.8,1
6,alcaldia_catalogo,16,130,0.10,IZTAPALAPA,16.2,GUSTAVO A. MADERO,CUAUHTEMOC,37.7,1
7,colonia_catalogo,1373,2718,2.19,Centro,1.5,Agricola Oriental,Agricola Pantitlan,3.7,1371


De esta tabla, podemos hacer varias anotaciones:
- De las 4 categorías, la más común fue la categoría D (Duplicado) donde un mismo incidente se registra más de una vez. Esto nos dice que el mecanismo de registro es muy susceptible de registrar multiples veces un mismo evento. 
- Aproximadamente el 90% de la base corresponde a accidentes, de los cuales el top 3 son choques sin lesionados, choques con lesionados y personas atropelladas.
- La alcalddía donde se inician más informes es la alcaldía Iztapalapa, seguida de la Gustavo A. Madero y la Cuauhtémoc.
- El 90% de los informes ingresan a través de una llamada al 911.

In [40]:
numericas = df[["longitud", "latitud", "duracion_h"]]

desc = numericas.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
desc["asimetría"] = numericas.skew()
display(desc.round(3))

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,asimetría
longitud,124261.0,-99.134,0.062,-99.353,-99.288,-99.233,-99.174,-99.137,-99.092,-99.029,-98.996,-98.948,-0.049
latitud,124261.0,19.381,0.072,19.095,19.203,19.261,19.333,19.383,19.434,19.493,19.526,19.632,-0.230
duracion_h,124261.0,4.394,5.007,0.006,0.329,3.004,3.069,3.128,3.191,10.368,25.161,76.669,4.440


Para las variables numéricas vemos la distribución para detectar algún valor extremo.
- Para las variables de latitud y longitud, vemos una desviación estándar pequeña, lo que es coherente con la base ya que todo sucede en la CDMX y un rango grande en esta variable podría indicar que se registran elementos de otros estados o incluso de otros países.
- Para la duración en horas, que es una variable construida, si observamos que el rango es muy amplio, llegando a tener elementos que se cierran en 77 horas. Sin embargo, esto podría deberse simplemente al proceso de cierre, más que a un fallo en el registro de la base, por lo que no podemos hacer suposiciones sin consultar con los expertos.

In [41]:
print("--- Rango de Fechas ---")
min_date = df['fecha_hora_creacion'].min()
max_date = df['fecha_hora_cierre'].max()
print(f"Fecha mínima: {min_date}")
print(f"Fecha máxima: {max_date}")

--- Rango de Fechas ---
Fecha mínima: 2021-12-31 20:59:59
Fecha máxima: 2024-02-29 23:58:48


Aquí vemos que las fechas van desde el 31 de diciembre de 2021 hasta el 29 de febrero de 2024.

Bajo esta estructura, cada fila representa un registro de incidente, mientras que las columnas describen sus características, como la clasificación, la ubicación y la información temporal disponible.